# UIT DSC 2026 LegalIR - Step 4 BGE-M3 + Metadata + RRF

GPU step: encode chunks with `BAAI/bge-m3`, retrieve dense candidates, fuse with Step 3 BM25 candidates by RRF, and create `submission.zip`.


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'FlagEmbedding'], check=True)


## Kaggle input paths


In [ ]:
from pathlib import Path
import argparse
import json
import subprocess
import sys
import zipfile

RAW_DATA_ROOT = Path('/kaggle/input/datasets/bowboochua9/stnhdscduaiti26')
LEGACY_STEP4_ROOT = RAW_DATA_ROOT / 'step4'
DATA_ZIP = LEGACY_STEP4_ROOT / 'step4.zip'
UNZIP_ROOT = Path('/kaggle/working/step4_input')

if (RAW_DATA_ROOT / 'best_config.json').exists():
    DATA_ROOT = RAW_DATA_ROOT
elif LEGACY_STEP4_ROOT.exists() and (LEGACY_STEP4_ROOT / 'best_config.json').exists():
    DATA_ROOT = LEGACY_STEP4_ROOT
elif DATA_ZIP.exists():
    UNZIP_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(UNZIP_ROOT)
    DATA_ROOT = UNZIP_ROOT
else:
    DATA_ROOT = RAW_DATA_ROOT

OUTPUT_DIR = Path('/kaggle/working/step4')
PUBLIC_FILE = Path('/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json')

required = [
    DATA_ROOT / 'chunks.jsonl',
    DATA_ROOT / 'train_split.json',
    DATA_ROOT / 'dev_split.json',
    DATA_ROOT / 'dev_rankings_best.jsonl',
    DATA_ROOT / 'public_rankings_best.jsonl',
    DATA_ROOT / 'best_config.json',
]
for path in required:
    assert path.exists(), path

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)


## Step 2 helpers: metrics, JSONL, and submission validation


In [ ]:
"""Step 2 Chunk-BM25 baseline for UIT DSC 2026 LegalIR.

Inputs are the data artifacts from Step 1:
- chunks.jsonl
- train_split.json
- dev_split.json

The implementation is dependency-free and CPU-only. It builds a BM25 inverted
index over chunks, retrieves top chunks, aggregates chunk scores to document
scores, then evaluates the document ranking on the dev split.
"""

from __future__ import annotations

import argparse
import heapq
import json
import math
import re
import statistics
import time
import zipfile
import unicodedata
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable


DEFAULT_KAGGLE_DATA_ROOT = Path("/kaggle/input/datasets/bowboochua9/stnhdscduaiti26")
DEFAULT_KAGGLE_PUBLIC_FILE = Path(
    "/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json"
)
DEFAULT_OUTPUT = (
    Path("/kaggle/working/step2")
    if Path("/kaggle").exists()
    else Path("task1/pipeline/step2/outputs")
)
MAX_SUBMISSION_DOCS = 5


STOPWORDS = {
    "a",
    "an",
    "anh",
    "ay",
    "bi",
    "boi",
    "cac",
    "can",
    "cho",
    "co",
    "con",
    "cua",
    "duoc",
    "da",
    "de",
    "den",
    "di",
    "do",
    "doi",
    "duoi",
    "gi",
    "hay",
    "hoac",
    "khi",
    "la",
    "lai",
    "lam",
    "mot",
    "nay",
    "neu",
    "nhu",
    "nhung",
    "o",
    "phai",
    "qua",
    "quy",
    "rieng",
    "sau",
    "se",
    "thi",
    "the",
    "theo",
    "thi",
    "trong",
    "tu",
    "va",
    "ve",
    "viec",
    "voi",
}


@dataclass(frozen=True)
class BM25Config:
    k1: float = 1.5
    b: float = 0.75
    top_chunks: int = 300
    top_docs: int = 100
    evidence_per_doc: int = 3
    aggregate_mean_top3_weight: float = 0.20
    aggregate_support_weight: float = 0.05
    heading_weight: float = 2.0
    use_deaccent: bool = True
    use_stopwords: bool = True


@dataclass
class ChunkMeta:
    chunk_id: str
    doc_id: str
    heading: str
    word_count: int
    is_empty_passage_fallback: bool


def read_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
        f.write("\n")


def write_submission_zip(submission_json: Path, submission_zip: Path) -> None:
    submission_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(
        submission_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
    ) as zf:
        zf.write(submission_json, arcname="submission.json")


def append_jsonl(path: Path, rows: Iterable[dict[str, Any]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, separators=(",", ":")))
            f.write("\n")
            count += 1
    return count


def strip_accents(text: str) -> str:
    text = text.replace("đ", "d").replace("Đ", "D")
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def tokenize(text: str, *, use_deaccent: bool, use_stopwords: bool) -> list[str]:
    if use_deaccent:
        text = strip_accents(text)
    text = text.lower()
    tokens = re.findall(r"[0-9a-zA-Z_]+", text)
    cleaned = []
    for token in tokens:
        if len(token) > 40:
            continue
        if use_stopwords and token in STOPWORDS:
            continue
        cleaned.append(token)
    return cleaned


def split_heading_body(text: str) -> tuple[str, str]:
    if "\n" not in text:
        return "", text
    heading, body = text.split("\n", 1)
    return heading.strip(), body.strip()


def find_step2_inputs(
    data_root: Path | None,
    chunks_file: Path | None,
    train_split_file: Path | None,
    dev_split_file: Path | None,
) -> tuple[Path, Path, Path]:
    root_candidates = []
    if data_root is not None:
        root_candidates.append(data_root)
    root_candidates.extend(
        [
            DEFAULT_KAGGLE_DATA_ROOT,
            Path("task1/pipeline/step1/outputs"),
            Path("task1/pipeline/step1/outputs/corpus"),
            Path("."),
        ]
    )

    resolved_chunks = chunks_file
    resolved_train = train_split_file
    resolved_dev = dev_split_file

    for root in root_candidates:
        chunk_candidates = [
            root / "chunks.jsonl",
            root / "corpus" / "chunks.jsonl",
        ]
        train_candidates = [
            root / "train_split.json",
            root / "splits" / "train_split.json",
        ]
        dev_candidates = [
            root / "dev_split.json",
            root / "splits" / "dev_split.json",
        ]
        if resolved_chunks is None:
            resolved_chunks = next((p for p in chunk_candidates if p.exists()), None)
        if resolved_train is None:
            resolved_train = next((p for p in train_candidates if p.exists()), None)
        if resolved_dev is None:
            resolved_dev = next((p for p in dev_candidates if p.exists()), None)

    missing = []
    if resolved_chunks is None or not resolved_chunks.exists():
        missing.append("chunks.jsonl")
    if resolved_train is None or not resolved_train.exists():
        missing.append("train_split.json")
    if resolved_dev is None or not resolved_dev.exists():
        missing.append("dev_split.json")
    if missing:
        raise FileNotFoundError(
            "Cannot locate Step 2 input(s): "
            + ", ".join(missing)
            + ". Pass --data-root or explicit file paths."
        )
    return resolved_chunks, resolved_train, resolved_dev


def find_public_file(public_file: Path | None, data_root: Path | None) -> Path | None:
    if public_file is not None:
        if not public_file.exists():
            raise FileNotFoundError(f"Cannot locate public file: {public_file}")
        return public_file

    candidates = []
    if data_root is not None:
        candidates.extend([data_root / "public-official.json", data_root / "public.json"])
    candidates.extend(
        [
            DEFAULT_KAGGLE_PUBLIC_FILE,
            Path("task1/public-official.json"),
            Path("public-official.json"),
        ]
    )
    return next((path for path in candidates if path.exists()), None)


class BM25ChunkIndex:
    def __init__(self, config: BM25Config) -> None:
        self.config = config
        self.postings: dict[str, list[tuple[int, float]]] = defaultdict(list)
        self.doc_freq: dict[str, int] = {}
        self.idf: dict[str, float] = {}
        self.chunk_lengths: list[float] = []
        self.chunk_meta: list[ChunkMeta] = []
        self.avgdl = 0.0

    def _chunk_token_weights(self, row: dict[str, Any]) -> Counter[str]:
        text = row.get("text", "")
        heading, body = split_heading_body(text if isinstance(text, str) else "")
        counter: Counter[str] = Counter(
            tokenize(body, use_deaccent=self.config.use_deaccent, use_stopwords=self.config.use_stopwords)
        )
        if heading:
            heading_counter = Counter(
                tokenize(
                    heading,
                    use_deaccent=self.config.use_deaccent,
                    use_stopwords=self.config.use_stopwords,
                )
            )
            for token, count in heading_counter.items():
                counter[token] += count * self.config.heading_weight
        return counter

    def build(
        self,
        chunks_file: Path,
        *,
        progress_every: int = 25000,
        limit_chunks: int = 0,
    ) -> dict[str, Any]:
        started = time.time()
        fallback_chunks = 0
        with chunks_file.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f):
                row = json.loads(line)
                metadata = row.get("metadata") if isinstance(row.get("metadata"), dict) else {}
                is_fallback = bool(metadata.get("is_empty_passage_fallback"))
                fallback_chunks += int(is_fallback)
                self.chunk_meta.append(
                    ChunkMeta(
                        chunk_id=str(row.get("chunk_id", idx)),
                        doc_id=str(row.get("doc_id", "")),
                        heading=str(row.get("heading", "")),
                        word_count=int(row.get("word_count") or 0),
                        is_empty_passage_fallback=is_fallback,
                    )
                )
                token_weights = self._chunk_token_weights(row)
                self.chunk_lengths.append(float(sum(token_weights.values())))
                for token, tf in token_weights.items():
                    self.postings[token].append((idx, float(tf)))
                if progress_every and (idx + 1) % progress_every == 0:
                    print(f"indexed {idx + 1:,} chunks; vocab={len(self.postings):,}")
                if limit_chunks and idx + 1 >= limit_chunks:
                    break

        num_chunks = len(self.chunk_meta)
        self.avgdl = statistics.fmean(self.chunk_lengths) if self.chunk_lengths else 0.0
        self.doc_freq = {token: len(posting) for token, posting in self.postings.items()}
        self.idf = {
            token: math.log(1.0 + (num_chunks - df + 0.5) / (df + 0.5))
            for token, df in self.doc_freq.items()
        }
        return {
            "num_chunks": num_chunks,
            "num_docs": len({meta.doc_id for meta in self.chunk_meta}),
            "num_terms": len(self.postings),
            "avgdl": self.avgdl,
            "fallback_chunks": fallback_chunks,
            "build_seconds": round(time.time() - started, 3),
        }

    def score_query_chunks(self, query: str) -> dict[int, float]:
        query_terms = Counter(
            tokenize(
                query,
                use_deaccent=self.config.use_deaccent,
                use_stopwords=self.config.use_stopwords,
            )
        )
        scores: defaultdict[int, float] = defaultdict(float)
        if not query_terms or not self.avgdl:
            return {}

        k1 = self.config.k1
        b = self.config.b
        for token, qtf in query_terms.items():
            posting = self.postings.get(token)
            if not posting:
                continue
            idf = self.idf[token]
            for chunk_idx, tf in posting:
                dl = self.chunk_lengths[chunk_idx]
                denom = tf + k1 * (1.0 - b + b * dl / self.avgdl)
                scores[chunk_idx] += qtf * idf * (tf * (k1 + 1.0) / denom)
        return scores

    def rank(self, query: str) -> list[dict[str, Any]]:
        chunk_scores = self.score_query_chunks(query)
        if not chunk_scores:
            return []

        top_chunk_items = heapq.nlargest(
            self.config.top_chunks, chunk_scores.items(), key=lambda item: item[1]
        )
        per_doc: dict[str, list[tuple[float, int]]] = defaultdict(list)
        for chunk_idx, score in top_chunk_items:
            doc_id = self.chunk_meta[chunk_idx].doc_id
            if doc_id:
                per_doc[doc_id].append((score, chunk_idx))

        doc_rows = []
        for doc_id, scored_chunks in per_doc.items():
            scored_chunks.sort(reverse=True)
            scores = [score for score, _ in scored_chunks]
            max_score = scores[0]
            mean_top3 = statistics.fmean(scores[:3])
            support_count = len(scored_chunks)
            doc_score = (
                max_score
                + self.config.aggregate_mean_top3_weight * mean_top3
                + self.config.aggregate_support_weight * support_count
            )
            evidence = []
            for score, chunk_idx in scored_chunks[: self.config.evidence_per_doc]:
                meta = self.chunk_meta[chunk_idx]
                evidence.append(
                    {
                        "chunk_id": meta.chunk_id,
                        "score": score,
                        "heading": meta.heading,
                        "word_count": meta.word_count,
                        "is_empty_passage_fallback": meta.is_empty_passage_fallback,
                    }
                )
            doc_rows.append(
                {
                    "doc_id": doc_id,
                    "score": doc_score,
                    "max_chunk_score": max_score,
                    "mean_top3_chunk_score": mean_top3,
                    "support_count": support_count,
                    "evidence": evidence,
                }
            )

        doc_rows.sort(key=lambda row: row["score"], reverse=True)
        return doc_rows[: self.config.top_docs]


def ranking_metrics_for_query(ranked_doc_ids: list[str], gold: list[str]) -> dict[str, float]:
    gold_set = {str(doc_id) for doc_id in gold}
    if not gold_set:
        return {
            "precision@5": 0.0,
            "recall@1": 0.0,
            "recall@5": 0.0,
            "recall@20": 0.0,
            "recall@50": 0.0,
            "recall@90": 0.0,
            "recall@100": 0.0,
            "hit@1": 0.0,
            "hit@5": 0.0,
            "hit@20": 0.0,
            "exist@90": 0.0,
            "mrr": 0.0,
        }

    def recall_at(k: int) -> float:
        return len(set(ranked_doc_ids[:k]) & gold_set) / len(gold_set)

    def hit_at(k: int) -> float:
        return 1.0 if set(ranked_doc_ids[:k]) & gold_set else 0.0

    top5 = ranked_doc_ids[:MAX_SUBMISSION_DOCS]
    precision5 = len(set(top5) & gold_set) / MAX_SUBMISSION_DOCS
    first_rank = next(
        (idx + 1 for idx, doc_id in enumerate(ranked_doc_ids) if doc_id in gold_set),
        None,
    )
    return {
        "precision@5": precision5,
        "recall@1": recall_at(1),
        "recall@5": recall_at(5),
        "recall@20": recall_at(20),
        "recall@50": recall_at(50),
        "recall@90": recall_at(90),
        "recall@100": recall_at(100),
        "hit@1": hit_at(1),
        "hit@5": hit_at(5),
        "hit@20": hit_at(20),
        "exist@90": hit_at(90),
        "mrr": 1.0 / first_rank if first_rank else 0.0,
    }


def evaluate_rankings(
    rankings: dict[str, list[str]],
    query_payload: dict[str, Any],
) -> dict[str, Any]:
    sums: Counter[str] = Counter()
    by_gold_count: dict[str, Counter[str]] = defaultdict(Counter)
    counts_by_gold_count: Counter[str] = Counter()
    per_query = {}

    for qid, row in query_payload.items():
        gold = row.get("answer", []) if isinstance(row, dict) else []
        metrics = ranking_metrics_for_query(rankings.get(str(qid), []), gold)
        per_query[str(qid)] = metrics
        sums.update(metrics)
        gold_count_key = str(len(gold))
        by_gold_count[gold_count_key].update(metrics)
        counts_by_gold_count[gold_count_key] += 1

    n = len(query_payload)
    macro = {key: (value / n if n else 0.0) for key, value in sorted(sums.items())}
    breakdown = {
        key: {
            metric: value / counts_by_gold_count[key]
            for metric, value in sorted(counter.items())
        }
        for key, counter in sorted(by_gold_count.items(), key=lambda item: int(item[0]))
    }
    return {
        "num_queries": n,
        "macro": macro,
        "by_gold_count": breakdown,
        "per_query": per_query,
    }


def make_submission(predictions_top5: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    return {
        str(qid): {"answer": [str(doc_id) for doc_id in doc_ids[:MAX_SUBMISSION_DOCS]]}
        for qid, doc_ids in predictions_top5.items()
    }


def validate_submission_payload(
    submission: Any,
    public_payload: dict[str, Any],
    valid_doc_ids: set[str],
) -> dict[str, Any]:
    issues = []
    if not isinstance(submission, dict):
        return {"num_errors": 1, "num_warnings": 0, "issues": ["root is not object"]}

    public_ids = {str(qid) for qid in public_payload}
    submission_ids = {str(qid) for qid in submission}
    for qid in sorted(public_ids - submission_ids)[:50]:
        issues.append({"severity": "error", "kind": "missing_query", "query_id": qid})
    for qid in sorted(submission_ids - public_ids)[:50]:
        issues.append({"severity": "error", "kind": "extra_query", "query_id": qid})

    answer_lengths: Counter[int] = Counter()
    for qid, row in submission.items():
        if not isinstance(row, dict):
            issues.append({"severity": "error", "kind": "row_not_object", "query_id": str(qid)})
            continue
        answer = row.get("answer")
        if not isinstance(answer, list):
            issues.append({"severity": "error", "kind": "answer_not_array", "query_id": str(qid)})
            continue
        answer_lengths[len(answer)] += 1
        if not (1 <= len(answer) <= MAX_SUBMISSION_DOCS):
            issues.append(
                {
                    "severity": "error",
                    "kind": "answer_count",
                    "query_id": str(qid),
                    "message": f"answer must contain 1-{MAX_SUBMISSION_DOCS} document IDs",
                }
            )
        normalized = [str(doc_id) for doc_id in answer]
        if any(not isinstance(doc_id, str) for doc_id in answer):
            issues.append({"severity": "error", "kind": "doc_id_not_string", "query_id": str(qid)})
        if len(normalized) != len(set(normalized)):
            issues.append({"severity": "error", "kind": "duplicate_doc_id", "query_id": str(qid)})
        for doc_id in normalized:
            if doc_id not in valid_doc_ids:
                issues.append(
                    {
                        "severity": "error",
                        "kind": "unknown_doc_id",
                        "query_id": str(qid),
                        "doc_id": doc_id,
                    }
                )

    return {
        "num_public_queries": len(public_ids),
        "num_submission_queries": len(submission_ids),
        "answer_length_distribution": dict(sorted(answer_lengths.items())),
        "num_errors": sum(issue["severity"] == "error" for issue in issues),
        "num_warnings": sum(issue["severity"] == "warning" for issue in issues),
        "issues": issues,
    }


def run_queries(
    index: BM25ChunkIndex,
    payload: dict[str, Any],
    *,
    output_rankings_file: Path,
    limit_queries: int = 0,
) -> tuple[dict[str, list[str]], dict[str, list[str]]]:
    rankings: dict[str, list[str]] = {}
    predictions_top5: dict[str, list[str]] = {}

    def rows() -> Iterable[dict[str, Any]]:
        started = time.time()
        query_items = list(payload.items())
        if limit_queries:
            query_items = query_items[:limit_queries]
        for pos, (qid, row) in enumerate(query_items, start=1):
            question = row.get("question", "") if isinstance(row, dict) else ""
            ranking_rows = index.rank(question)
            doc_ids = [item["doc_id"] for item in ranking_rows]
            rankings[str(qid)] = doc_ids
            predictions_top5[str(qid)] = doc_ids[:MAX_SUBMISSION_DOCS]
            if pos % 100 == 0:
                print(f"ranked {pos:,}/{len(query_items):,} queries in {time.time() - started:.1f}s")
            yield {
                "query_id": str(qid),
                "question": question,
                "gold": row.get("answer", []) if isinstance(row, dict) else [],
                "top_docs": ranking_rows,
            }

    append_jsonl(output_rankings_file, rows())
    return rankings, predictions_top5


def run_step2(args: argparse.Namespace) -> dict[str, Any]:
    chunks_file, train_split_file, dev_split_file = find_step2_inputs(
        args.data_root, args.chunks_file, args.train_split_file, args.dev_split_file
    )
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    config = BM25Config(
        k1=args.k1,
        b=args.b,
        top_chunks=args.top_chunks,
        top_docs=args.top_docs,
        evidence_per_doc=args.evidence_per_doc,
        aggregate_mean_top3_weight=args.mean_top3_weight,
        aggregate_support_weight=args.support_weight,
        heading_weight=args.heading_weight,
        use_deaccent=not args.no_deaccent,
        use_stopwords=not args.no_stopwords,
    )

    print("loading query splits")
    train_payload = read_json(train_split_file)
    dev_payload = read_json(dev_split_file)
    print("building BM25 chunk index")
    index = BM25ChunkIndex(config)
    index_stats = index.build(
        chunks_file,
        progress_every=args.progress_every,
        limit_chunks=args.limit_chunks,
    )

    print("ranking dev queries")
    dev_rankings, dev_top5 = run_queries(
        index,
        dev_payload,
        output_rankings_file=output_dir / "rankings" / "dev_rankings.jsonl",
        limit_queries=args.limit_queries,
    )
    dev_eval_payload = dict(list(dev_payload.items())[: args.limit_queries]) if args.limit_queries else dev_payload
    dev_metrics = evaluate_rankings(dev_rankings, dev_eval_payload)
    write_json(output_dir / "predictions" / "dev_predictions_top5.json", dev_top5)
    write_json(output_dir / "metrics" / "dev_metrics.json", dev_metrics)

    train_metrics = None
    if args.eval_train:
        print("ranking train queries")
        train_rankings, train_top5 = run_queries(
            index,
            train_payload,
            output_rankings_file=output_dir / "rankings" / "train_rankings.jsonl",
            limit_queries=args.limit_queries,
        )
        train_eval_payload = dict(list(train_payload.items())[: args.limit_queries]) if args.limit_queries else train_payload
        train_metrics = evaluate_rankings(train_rankings, train_eval_payload)
        write_json(output_dir / "predictions" / "train_predictions_top5.json", train_top5)
        write_json(output_dir / "metrics" / "train_metrics.json", train_metrics)

    public_outputs = None
    if args.predict_public:
        public_file = find_public_file(args.public_file, args.data_root)
        if public_file is None:
            raise FileNotFoundError(
                "Cannot locate public-official.json. Pass --public-file when using --predict-public."
            )
        print("ranking public queries")
        public_payload = read_json(public_file)
        _, public_top5 = run_queries(
            index,
            public_payload,
            output_rankings_file=output_dir / "rankings" / "public_rankings.jsonl",
            limit_queries=args.limit_queries,
        )
        public_eval_payload = (
            dict(list(public_payload.items())[: args.limit_queries])
            if args.limit_queries
            else public_payload
        )
        submission = make_submission(public_top5)
        submission_json = output_dir / "submission" / "submission.json"
        submission_zip = output_dir / "submission" / args.submission_zip_name
        write_json(submission_json, submission)
        submission_report = validate_submission_payload(
            submission,
            public_eval_payload,
            {meta.doc_id for meta in index.chunk_meta},
        )
        write_json(output_dir / "submission" / "submission_validation.json", submission_report)
        if submission_report["num_errors"]:
            raise ValueError(
                f"Submission validation failed with {submission_report['num_errors']} errors. "
                f"See {output_dir / 'submission' / 'submission_validation.json'}"
            )
        write_submission_zip(submission_json, submission_zip)
        public_outputs = {
            "public_file": str(public_file),
            "public_rankings": "rankings/public_rankings.jsonl",
            "submission_json": "submission/submission.json",
            "submission_zip": f"submission/{args.submission_zip_name}",
            "submission_validation": "submission/submission_validation.json",
        }

    run_report = {
        "inputs": {
            "chunks_file": str(chunks_file),
            "train_split_file": str(train_split_file),
            "dev_split_file": str(dev_split_file),
            "output_dir": str(output_dir),
        },
        "config": asdict(config),
        "index": index_stats,
        "dev_macro": dev_metrics["macro"],
        "train_macro": train_metrics["macro"] if train_metrics else None,
        "outputs": {
            "dev_rankings": "rankings/dev_rankings.jsonl",
            "dev_predictions_top5": "predictions/dev_predictions_top5.json",
            "dev_metrics": "metrics/dev_metrics.json",
        },
        "public_outputs": public_outputs,
    }
    write_json(output_dir / "reports" / "run_report.json", run_report)
    return run_report


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, default=None)
    parser.add_argument("--chunks-file", type=Path, default=None)
    parser.add_argument("--train-split-file", type=Path, default=None)
    parser.add_argument("--dev-split-file", type=Path, default=None)
    parser.add_argument("--public-file", type=Path, default=None)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--k1", type=float, default=1.5)
    parser.add_argument("--b", type=float, default=0.75)
    parser.add_argument("--top-chunks", type=int, default=300)
    parser.add_argument("--top-docs", type=int, default=100)
    parser.add_argument("--evidence-per-doc", type=int, default=3)
    parser.add_argument("--mean-top3-weight", type=float, default=0.20)
    parser.add_argument("--support-weight", type=float, default=0.05)
    parser.add_argument("--heading-weight", type=float, default=2.0)
    parser.add_argument("--no-deaccent", action="store_true")
    parser.add_argument("--no-stopwords", action="store_true")
    parser.add_argument("--eval-train", action="store_true")
    parser.add_argument("--predict-public", action="store_true")
    parser.add_argument("--submission-zip-name", default="submission.zip")
    parser.add_argument("--progress-every", type=int, default=25000)
    parser.add_argument("--limit-chunks", type=int, default=0, help="Debug only.")
    parser.add_argument("--limit-queries", type=int, default=0, help="Debug only.")
    return parser


def main() -> None:
    args = build_arg_parser().parse_args()
    report = run_step2(args)
    print(json.dumps(report["dev_macro"], ensure_ascii=False, indent=2))
    print(f"Wrote outputs to: {args.output_dir}")



## Step 4: BGE-M3 dense retrieval, metadata rank, and RRF fusion


In [ ]:
"""Step 4 BGE-M3 dense retrieval + metadata branch + RRF.

Designed for Kaggle T4x2:
- downloads/loads BAAI/bge-m3 locally through FlagEmbedding
- encodes Step 1 chunks into cached fp16 dense vectors
- retrieves top dense chunks for dev/public queries
- aggregates chunks to document candidates
- fuses BM25 + dense + optional metadata branch via Reciprocal Rank Fusion
- evaluates on dev and can package public submission.zip

No hosted inference/API is used. Model weights are loaded into the notebook
runtime and all inference is local.
"""

from __future__ import annotations

import argparse
import json
import re
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable

import numpy as np



DEFAULT_DATA_ROOT = Path("/kaggle/input/datasets/bowboochua9/stnhdscduaiti26")
DEFAULT_STEP4_INPUT_ROOT = DEFAULT_DATA_ROOT / "step4"
DEFAULT_OUTPUT = (
    Path("/kaggle/working/step4")
    if Path("/kaggle").exists()
    else Path("task1/pipeline/step4/outputs")
)
DEFAULT_PUBLIC_FILE = Path(
    "/kaggle/input/datasets/ttdatto/uit-dsc26/LegalIR - Public Test/public-official.json"
)


@dataclass(frozen=True)
class Step4Config:
    model_name: str = "BAAI/bge-m3"
    batch_size: int = 12
    query_batch_size: int = 32
    max_length: int = 512
    use_fp16: bool = True
    dense_top_chunks: int = 300
    dense_top_docs: int = 100
    evidence_per_doc: int = 3
    aggregate_mean_top3_weight: float = 0.20
    aggregate_support_weight: float = 0.05
    rrf_k: int = 60
    bm25_weight: float = 1.0
    dense_weight: float = 1.0
    metadata_weight: float = 0.20
    fused_top_docs: int = 100
    search_block_size: int = 32768


def find_step4_inputs(
    *,
    data_root: Path | None,
    chunks_file: Path | None,
    train_split_file: Path | None,
    dev_split_file: Path | None,
    bm25_dev_rankings_file: Path | None,
    best_config_file: Path | None,
) -> tuple[Path, Path, Path, Path, Path | None]:
    roots = []
    if data_root is not None:
        roots.append(data_root)
    roots.extend(
        [
            DEFAULT_STEP4_INPUT_ROOT,
            DEFAULT_DATA_ROOT,
            Path("task1/pipeline/step4"),
            Path("task1/pipeline/step1/outputs"),
            Path("."),
        ]
    )

    resolved_chunks = chunks_file
    resolved_train = train_split_file
    resolved_dev = dev_split_file
    resolved_bm25 = bm25_dev_rankings_file
    resolved_best_config = best_config_file

    for root in roots:
        if resolved_chunks is None:
            resolved_chunks = next(
                (
                    p
                    for p in [
                        root / "step1" / "chunks.jsonl",
                        root / "chunks.jsonl",
                        root / "corpus" / "chunks.jsonl",
                    ]
                    if p.exists()
                ),
                None,
            )
        if resolved_train is None:
            resolved_train = next(
                (
                    p
                    for p in [
                        root / "step1" / "train_split.json",
                        root / "train_split.json",
                        root / "splits" / "train_split.json",
                    ]
                    if p.exists()
                ),
                None,
            )
        if resolved_dev is None:
            resolved_dev = next(
                (
                    p
                    for p in [
                        root / "step1" / "dev_split.json",
                        root / "dev_split.json",
                        root / "splits" / "dev_split.json",
                    ]
                    if p.exists()
                ),
                None,
            )
        if resolved_bm25 is None:
            resolved_bm25 = next(
                (
                    p
                    for p in [
                        root / "step3" / "dev_rankings_best.jsonl",
                        root / "dev_rankings_best.jsonl",
                        root / "rankings" / "dev_rankings_best.jsonl",
                        Path("task1/pipeline/step3/outputs/rankings/dev_rankings_best.jsonl"),
                    ]
                    if p.exists()
                ),
                None,
            )
        if resolved_best_config is None:
            resolved_best_config = next(
                (
                    p
                    for p in [
                        root / "step3" / "best_config.json",
                        root / "best_config.json",
                        root / "configs" / "best_config.json",
                        Path("task1/pipeline/step3/outputs/configs/best_config.json"),
                    ]
                    if p.exists()
                ),
                None,
            )

    missing = []
    if resolved_chunks is None or not resolved_chunks.exists():
        missing.append("chunks.jsonl")
    if resolved_train is None or not resolved_train.exists():
        missing.append("train_split.json")
    if resolved_dev is None or not resolved_dev.exists():
        missing.append("dev_split.json")
    if resolved_bm25 is None or not resolved_bm25.exists():
        missing.append("dev_rankings_best.jsonl")
    if missing:
        raise FileNotFoundError("Missing Step 4 input(s): " + ", ".join(missing))
    return resolved_chunks, resolved_train, resolved_dev, resolved_bm25, resolved_best_config


def iter_jsonl(path: Path) -> Iterable[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def load_chunks(chunks_file: Path, *, limit_chunks: int = 0) -> tuple[list[dict[str, Any]], dict[str, dict[str, Any]]]:
    chunks = []
    doc_metadata: dict[str, dict[str, Any]] = {}
    for idx, row in enumerate(iter_jsonl(chunks_file)):
        metadata = row.get("metadata") if isinstance(row.get("metadata"), dict) else {}
        chunk = {
            "chunk_idx": idx,
            "chunk_id": str(row.get("chunk_id", idx)),
            "doc_id": str(row.get("doc_id", "")),
            "text": str(row.get("text", "")),
            "heading": str(row.get("heading", "")),
            "word_count": int(row.get("word_count") or 0),
            "metadata": metadata,
        }
        chunks.append(chunk)
        if chunk["doc_id"] and chunk["doc_id"] not in doc_metadata:
            doc_metadata[chunk["doc_id"]] = metadata | {"heading": chunk["heading"]}
        if limit_chunks and idx + 1 >= limit_chunks:
            break
    return chunks, doc_metadata


def load_bm25_rankings(path: Path, *, limit_queries: int = 0) -> dict[str, list[dict[str, Any]]]:
    rankings = {}
    for idx, row in enumerate(iter_jsonl(path)):
        rankings[str(row["query_id"])] = row.get("top_docs", [])
        if limit_queries and idx + 1 >= limit_queries:
            break
    return rankings


def import_bge_model() -> Any:
    try:
        from FlagEmbedding import BGEM3FlagModel
    except ImportError as exc:  # pragma: no cover - depends on Kaggle env
        raise ImportError(
            "FlagEmbedding is required. In Kaggle run: pip install -U FlagEmbedding"
        ) from exc
    return BGEM3FlagModel


def encode_texts(model: Any, texts: list[str], *, batch_size: int, max_length: int) -> np.ndarray:
    output = model.encode(
        texts,
        batch_size=batch_size,
        max_length=max_length,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    vectors = output["dense_vecs"] if isinstance(output, dict) else output
    vectors = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.maximum(norms, 1e-12)
    return vectors


def build_or_load_embeddings(
    *,
    chunks: list[dict[str, Any]],
    output_dir: Path,
    config: Step4Config,
    force_rebuild: bool,
) -> tuple[np.ndarray, Path]:
    emb_path = output_dir / "embeddings" / "chunk_embeddings_fp16.npy"
    meta_path = output_dir / "embeddings" / "chunk_embedding_meta.json"
    if emb_path.exists() and meta_path.exists() and not force_rebuild:
        print(f"loading cached embeddings: {emb_path}")
        return np.load(emb_path, mmap_mode="r"), emb_path

    BGEM3FlagModel = import_bge_model()
    model = BGEM3FlagModel(config.model_name, use_fp16=config.use_fp16)
    emb_path.parent.mkdir(parents=True, exist_ok=True)

    all_vecs = []
    started = time.time()
    for start in range(0, len(chunks), config.batch_size):
        batch = chunks[start : start + config.batch_size]
        vecs = encode_texts(
            model,
            [row["text"] for row in batch],
            batch_size=config.batch_size,
            max_length=config.max_length,
        )
        all_vecs.append(vecs.astype(np.float16))
        if (start // config.batch_size + 1) % 100 == 0:
            print(f"encoded {min(start + config.batch_size, len(chunks)):,}/{len(chunks):,} chunks")
    embeddings = np.vstack(all_vecs)
    np.save(emb_path, embeddings)
    write_json(
        meta_path,
        {
            "model_name": config.model_name,
            "num_chunks": len(chunks),
            "dim": int(embeddings.shape[1]),
            "dtype": "float16",
            "max_length": config.max_length,
            "seconds": round(time.time() - started, 3),
        },
    )
    return np.load(emb_path, mmap_mode="r"), emb_path


def dense_search_torch(
    *,
    chunk_embeddings: np.ndarray,
    query_embeddings: np.ndarray,
    top_k: int,
    block_size: int,
) -> list[list[tuple[int, float]]]:
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    query_tensor = torch.tensor(query_embeddings, dtype=torch.float16 if device == "cuda" else torch.float32, device=device)
    results: list[list[tuple[int, float]]] = []
    for q_idx in range(query_tensor.shape[0]):
        top_scores = None
        top_indices = None
        q = query_tensor[q_idx : q_idx + 1].T
        for start in range(0, chunk_embeddings.shape[0], block_size):
            block_np = np.asarray(chunk_embeddings[start : start + block_size], dtype=np.float16 if device == "cuda" else np.float32)
            block = torch.tensor(block_np, device=device)
            scores = (block @ q).squeeze(1)
            block_k = min(top_k, scores.numel())
            vals, idxs = torch.topk(scores, k=block_k)
            idxs = idxs + start
            if top_scores is None:
                top_scores, top_indices = vals, idxs
            else:
                top_scores = torch.cat([top_scores, vals])
                top_indices = torch.cat([top_indices, idxs])
                vals2, order = torch.topk(top_scores, k=min(top_k, top_scores.numel()))
                top_indices = top_indices[order]
                top_scores = vals2
        results.append(
            [
                (int(idx), float(score))
                for idx, score in zip(top_indices.detach().cpu().tolist(), top_scores.detach().cpu().tolist())
            ]
        )
    return results


def aggregate_dense_docs(
    chunks: list[dict[str, Any]],
    chunk_hits: list[tuple[int, float]],
    config: Step4Config,
) -> list[dict[str, Any]]:
    per_doc: dict[str, list[tuple[float, int]]] = defaultdict(list)
    for chunk_idx, score in chunk_hits[: config.dense_top_chunks]:
        doc_id = chunks[chunk_idx]["doc_id"]
        if doc_id:
            per_doc[doc_id].append((score, chunk_idx))

    rows = []
    for doc_id, scored in per_doc.items():
        scored.sort(reverse=True)
        scores = [s for s, _ in scored]
        max_score = scores[0]
        mean_top3 = sum(scores[:3]) / min(3, len(scores))
        support_count = len(scored)
        doc_score = (
            max_score
            + config.aggregate_mean_top3_weight * mean_top3
            + config.aggregate_support_weight * support_count
        )
        evidence = []
        for score, chunk_idx in scored[: config.evidence_per_doc]:
            row = chunks[chunk_idx]
            evidence.append(
                {
                    "chunk_id": row["chunk_id"],
                    "score": score,
                    "heading": row["heading"],
                    "word_count": row["word_count"],
                    "is_empty_passage_fallback": bool(row["metadata"].get("is_empty_passage_fallback")),
                }
            )
        rows.append(
            {
                "doc_id": doc_id,
                "score": doc_score,
                "max_chunk_score": max_score,
                "mean_top3_chunk_score": mean_top3,
                "support_count": support_count,
                "evidence": evidence,
            }
        )
    rows.sort(key=lambda row: row["score"], reverse=True)
    return rows[: config.dense_top_docs]


def extract_query_metadata_signals(question: str) -> dict[str, set[str]]:
    q = question.lower()
    years = set(re.findall(r"\b(19\d{2}|20\d{2})\b", q))
    doc_types = set()
    for label, patterns in {
        "luat": ["luật", "bộ luật"],
        "nghi_dinh": ["nghị định"],
        "thong_tu": ["thông tư"],
        "quyet_dinh": ["quyết định"],
        "qcvn": ["qcvn", "quy chuẩn"],
        "tcvn": ["tcvn", "tiêu chuẩn"],
    }.items():
        if any(p in q for p in patterns):
            doc_types.add(label)
    numbers = set(re.findall(r"\b\d{1,5}(?:/\d{4})?(?:/[a-zA-ZĐđ0-9.-]+)?\b", question))
    return {"years": years, "doc_types": doc_types, "numbers": numbers}


def metadata_rank(
    question: str,
    candidate_doc_ids: list[str],
    doc_metadata: dict[str, dict[str, Any]],
) -> list[str]:
    signals = extract_query_metadata_signals(question)
    scored = []
    for doc_id in candidate_doc_ids:
        meta = doc_metadata.get(doc_id, {})
        score = 0.0
        years = set(str(y) for y in meta.get("years", []))
        if signals["years"] & years:
            score += 2.0 * len(signals["years"] & years)
        if meta.get("doc_type") in signals["doc_types"]:
            score += 2.0
        hay = " ".join(str(meta.get(key, "")) for key in ["issue_number", "heading"]).lower()
        score += sum(1.0 for number in signals["numbers"] if number and number.lower() in hay)
        if score > 0:
            scored.append((doc_id, score))
    scored.sort(key=lambda item: item[1], reverse=True)
    return [doc_id for doc_id, _ in scored]


def rrf_fuse(
    branches: list[tuple[str, list[str], float]],
    *,
    rrf_k: int,
    top_docs: int,
) -> list[str]:
    scores: defaultdict[str, float] = defaultdict(float)
    first_seen: dict[str, int] = {}
    for _, doc_ids, weight in branches:
        for rank, doc_id in enumerate(doc_ids, start=1):
            scores[doc_id] += weight / (rrf_k + rank)
            first_seen.setdefault(doc_id, len(first_seen))
    return [
        doc_id
        for doc_id, _ in sorted(
            scores.items(),
            key=lambda item: (-item[1], first_seen[item[0]]),
        )[:top_docs]
    ]


def run_queries(
    *,
    model: Any,
    chunk_embeddings: np.ndarray,
    chunks: list[dict[str, Any]],
    doc_metadata: dict[str, dict[str, Any]],
    payload: dict[str, Any],
    bm25_rankings: dict[str, list[dict[str, Any]]] | None,
    output_rankings_file: Path,
    config: Step4Config,
    limit_queries: int = 0,
) -> tuple[dict[str, list[str]], dict[str, list[str]]]:
    query_items = list(payload.items())
    if limit_queries:
        query_items = query_items[:limit_queries]
    questions = [row.get("question", "") if isinstance(row, dict) else "" for _, row in query_items]
    print(f"encoding {len(questions):,} queries")
    query_embeddings = encode_texts(
        model,
        questions,
        batch_size=config.query_batch_size,
        max_length=config.max_length,
    )
    print("dense searching queries")
    dense_hits = dense_search_torch(
        chunk_embeddings=chunk_embeddings,
        query_embeddings=query_embeddings,
        top_k=config.dense_top_chunks,
        block_size=config.search_block_size,
    )
    rankings: dict[str, list[str]] = {}
    predictions: dict[str, list[str]] = {}

    def rows() -> Iterable[dict[str, Any]]:
        for idx, ((qid, row), chunk_hits) in enumerate(zip(query_items, dense_hits), start=1):
            question = row.get("question", "") if isinstance(row, dict) else ""
            dense_docs = aggregate_dense_docs(chunks, chunk_hits, config)
            dense_ids = [doc["doc_id"] for doc in dense_docs]
            bm25_docs = (bm25_rankings or {}).get(str(qid), [])
            bm25_ids = [str(doc["doc_id"]) for doc in bm25_docs]
            union = list(dict.fromkeys(bm25_ids + dense_ids))
            metadata_ids = metadata_rank(question, union, doc_metadata)
            fused_ids = rrf_fuse(
                [
                    ("bm25", bm25_ids, config.bm25_weight),
                    ("dense", dense_ids, config.dense_weight),
                    ("metadata", metadata_ids, config.metadata_weight),
                ],
                rrf_k=config.rrf_k,
                top_docs=config.fused_top_docs,
            )
            rankings[str(qid)] = fused_ids
            predictions[str(qid)] = fused_ids[:MAX_SUBMISSION_DOCS]
            if idx % 100 == 0:
                print(f"fused {idx:,}/{len(query_items):,} queries")
            yield {
                "query_id": str(qid),
                "question": question,
                "gold": row.get("answer", []) if isinstance(row, dict) else [],
                "bm25_top_docs": bm25_docs[: config.fused_top_docs],
                "dense_top_docs": dense_docs,
                "metadata_ranked_doc_ids": metadata_ids,
                "fused_doc_ids": fused_ids,
            }

    append_jsonl(output_rankings_file, rows())
    return rankings, predictions


def run_step4(args: argparse.Namespace) -> dict[str, Any]:
    chunks_file, train_file, dev_file, bm25_dev_file, best_config_file = find_step4_inputs(
        data_root=args.data_root,
        chunks_file=args.chunks_file,
        train_split_file=args.train_split_file,
        dev_split_file=args.dev_split_file,
        bm25_dev_rankings_file=args.bm25_dev_rankings_file,
        best_config_file=args.best_config_file,
    )
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    config = Step4Config(
        model_name=args.model_name,
        batch_size=args.batch_size,
        query_batch_size=args.query_batch_size,
        max_length=args.max_length,
        use_fp16=not args.no_fp16,
        dense_top_chunks=args.dense_top_chunks,
        dense_top_docs=args.dense_top_docs,
        evidence_per_doc=args.evidence_per_doc,
        rrf_k=args.rrf_k,
        bm25_weight=args.bm25_weight,
        dense_weight=args.dense_weight,
        metadata_weight=args.metadata_weight,
        fused_top_docs=args.fused_top_docs,
        search_block_size=args.search_block_size,
    )
    write_json(output_dir / "configs" / "step4_config.json", asdict(config))

    print("loading chunks")
    chunks, doc_metadata = load_chunks(chunks_file, limit_chunks=args.limit_chunks)
    print(f"loaded {len(chunks):,} chunks; docs={len(doc_metadata):,}")
    embeddings, emb_path = build_or_load_embeddings(
        chunks=chunks,
        output_dir=output_dir,
        config=config,
        force_rebuild=args.force_rebuild_embeddings,
    )

    BGEM3FlagModel = import_bge_model()
    model = BGEM3FlagModel(config.model_name, use_fp16=config.use_fp16)

    dev_payload = read_json(dev_file)
    bm25_dev = load_bm25_rankings(bm25_dev_file, limit_queries=args.limit_queries)
    dev_rankings, dev_predictions = run_queries(
        model=model,
        chunk_embeddings=embeddings,
        chunks=chunks,
        doc_metadata=doc_metadata,
        payload=dev_payload,
        bm25_rankings=bm25_dev,
        output_rankings_file=output_dir / "rankings" / "dev_rankings_rrf.jsonl",
        config=config,
        limit_queries=args.limit_queries,
    )
    dev_eval_payload = dict(list(dev_payload.items())[: args.limit_queries]) if args.limit_queries else dev_payload
    dev_metrics = evaluate_rankings(dev_rankings, dev_eval_payload)
    write_json(output_dir / "metrics" / "dev_metrics_rrf.json", dev_metrics)
    write_json(output_dir / "predictions" / "dev_predictions_top5_rrf.json", dev_predictions)

    train_metrics = None
    if args.eval_train:
        train_payload = read_json(train_file)
        bm25_train = (
            load_bm25_rankings(args.bm25_train_rankings_file, limit_queries=args.limit_queries)
            if args.bm25_train_rankings_file
            else None
        )
        train_rankings, train_predictions = run_queries(
            model=model,
            chunk_embeddings=embeddings,
            chunks=chunks,
            doc_metadata=doc_metadata,
            payload=train_payload,
            bm25_rankings=bm25_train,
            output_rankings_file=output_dir / "rankings" / "train_rankings_rrf.jsonl",
            config=config,
            limit_queries=args.limit_queries,
        )
        train_eval_payload = dict(list(train_payload.items())[: args.limit_queries]) if args.limit_queries else train_payload
        train_metrics = evaluate_rankings(train_rankings, train_eval_payload)
        write_json(output_dir / "metrics" / "train_metrics_rrf.json", train_metrics)
        write_json(output_dir / "predictions" / "train_predictions_top5_rrf.json", train_predictions)

    public_outputs = None
    if args.predict_public:
        public_file = find_public_file(args.public_file, args.data_root) or DEFAULT_PUBLIC_FILE
        if not public_file.exists():
            raise FileNotFoundError(f"Cannot locate public file: {public_file}")
        public_payload = read_json(public_file)
        # For public we do not have BM25 public rankings in the Step 4 minimal
        # input set, so dense+metadata is used unless --bm25-public-rankings-file is passed.
        bm25_public = (
            load_bm25_rankings(args.bm25_public_rankings_file, limit_queries=args.limit_queries)
            if args.bm25_public_rankings_file
            else None
        )
        public_rankings, public_predictions = run_queries(
            model=model,
            chunk_embeddings=embeddings,
            chunks=chunks,
            doc_metadata=doc_metadata,
            payload=public_payload,
            bm25_rankings=bm25_public,
            output_rankings_file=output_dir / "rankings" / "public_rankings_rrf.jsonl",
            config=config,
            limit_queries=args.limit_queries,
        )
        submission = make_submission(public_predictions)
        public_eval_payload = dict(list(public_payload.items())[: args.limit_queries]) if args.limit_queries else public_payload
        validation = validate_submission_payload(
            submission,
            public_eval_payload,
            {chunk["doc_id"] for chunk in chunks},
        )
        submission_dir = output_dir / "submission"
        write_json(submission_dir / "submission.json", submission)
        write_json(submission_dir / "submission_validation.json", validation)
        if validation["num_errors"]:
            raise ValueError(f"Submission validation failed: {validation['num_errors']} errors")
        write_submission_zip(submission_dir / "submission.json", submission_dir / "submission.zip")
        public_outputs = {
            "rankings": "rankings/public_rankings_rrf.jsonl",
            "submission_zip": "submission/submission.zip",
            "submission_validation": "submission/submission_validation.json",
        }

    report = {
        "inputs": {
            "chunks_file": str(chunks_file),
            "train_split_file": str(train_file),
            "dev_split_file": str(dev_file),
            "bm25_dev_rankings_file": str(bm25_dev_file),
            "best_config_file": str(best_config_file) if best_config_file else None,
        },
        "config": asdict(config),
        "embeddings_file": str(emb_path),
        "dev_macro": dev_metrics["macro"],
        "train_macro": train_metrics["macro"] if train_metrics else None,
        "public_outputs": public_outputs,
        "next_gpu_steps": {
            "step5_biencoder_finetune": [
                "chunks.jsonl",
                "train_split.json",
                "dev_split.json",
                "rankings/dev_rankings_rrf.jsonl",
                "optional rankings/train_rankings_rrf.jsonl if mined on train",
            ]
        },
        "sources": {
            "bge_m3_model_card": "https://huggingface.co/BAAI/bge-m3"
        },
    }
    write_json(output_dir / "reports" / "run_report.json", report)
    return report


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, default=None)
    parser.add_argument("--chunks-file", type=Path, default=None)
    parser.add_argument("--train-split-file", type=Path, default=None)
    parser.add_argument("--dev-split-file", type=Path, default=None)
    parser.add_argument("--bm25-dev-rankings-file", type=Path, default=None)
    parser.add_argument("--bm25-train-rankings-file", type=Path, default=None)
    parser.add_argument("--bm25-public-rankings-file", type=Path, default=None)
    parser.add_argument("--best-config-file", type=Path, default=None)
    parser.add_argument("--public-file", type=Path, default=None)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--model-name", default="BAAI/bge-m3")
    parser.add_argument("--batch-size", type=int, default=12)
    parser.add_argument("--query-batch-size", type=int, default=32)
    parser.add_argument("--max-length", type=int, default=512)
    parser.add_argument("--no-fp16", action="store_true")
    parser.add_argument("--dense-top-chunks", type=int, default=300)
    parser.add_argument("--dense-top-docs", type=int, default=100)
    parser.add_argument("--evidence-per-doc", type=int, default=3)
    parser.add_argument("--rrf-k", type=int, default=60)
    parser.add_argument("--bm25-weight", type=float, default=1.0)
    parser.add_argument("--dense-weight", type=float, default=1.0)
    parser.add_argument("--metadata-weight", type=float, default=0.20)
    parser.add_argument("--fused-top-docs", type=int, default=100)
    parser.add_argument("--search-block-size", type=int, default=32768)
    parser.add_argument("--predict-public", action="store_true")
    parser.add_argument("--eval-train", action="store_true")
    parser.add_argument("--force-rebuild-embeddings", action="store_true")
    parser.add_argument("--limit-chunks", type=int, default=0, help="Debug only.")
    parser.add_argument("--limit-queries", type=int, default=0, help="Debug only.")
    return parser


def main() -> None:
    args = build_arg_parser().parse_args()
    report = run_step4(args)
    print(json.dumps(report["dev_macro"], ensure_ascii=False, indent=2))
    print(f"Wrote outputs to: {args.output_dir}")



## Run Step 4 and create public submission


In [ ]:
args = argparse.Namespace(
    data_root=DATA_ROOT,
    chunks_file=None,
    train_split_file=None,
    dev_split_file=None,
    bm25_dev_rankings_file=None,
    bm25_train_rankings_file=None,
    bm25_public_rankings_file=DATA_ROOT / 'public_rankings_best.jsonl',
    best_config_file=None,
    public_file=PUBLIC_FILE,
    output_dir=OUTPUT_DIR,
    model_name='BAAI/bge-m3',
    batch_size=12,
    query_batch_size=32,
    max_length=512,
    no_fp16=False,
    dense_top_chunks=300,
    dense_top_docs=100,
    evidence_per_doc=3,
    rrf_k=60,
    bm25_weight=1.0,
    dense_weight=1.0,
    metadata_weight=0.20,
    fused_top_docs=100,
    search_block_size=32768,
    predict_public=True,
    eval_train=False,
    force_rebuild_embeddings=False,
    limit_chunks=0,
    limit_queries=0,
)

report = run_step4(args)
print(json.dumps(report['dev_macro'], ensure_ascii=False, indent=2))
print(f"Wrote outputs to: {args.output_dir}")
print('Submission:', args.output_dir / 'submission' / 'submission.zip')


## Files to download


In [ ]:
for path in [
    OUTPUT_DIR / 'metrics' / 'dev_metrics_rrf.json',
    OUTPUT_DIR / 'rankings' / 'dev_rankings_rrf.jsonl',
    OUTPUT_DIR / 'rankings' / 'public_rankings_rrf.jsonl',
    OUTPUT_DIR / 'submission' / 'submission.zip',
    OUTPUT_DIR / 'reports' / 'run_report.json',
]:
    print(path, 'OK' if path.exists() else 'MISSING')
